# Binary Classification with CNNs: Chest X-Ray Images (Pneumonia)

## Setup

In [ ]:
import numpy as np
import tensorflow as tf


import datetime

notebook_start_time = datetime.datetime.now()

In [ ]:
test_dir = "data/test"
train_dir = "data/train"
validation_dir = "data/val"

height, width, channels = 150, 150, 1

batch_size = 32

# While working, I discovered that the tf.keras.preprocessing.image.ImageDataGenerator class is deprecated
# using this new API instead

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    color_mode="grayscale",
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    color_mode="grayscale",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(height, width),
    batch_size=batch_size,
    label_mode="int",
    color_mode="grayscale",
    shuffle=False,
)

# because the outputs are batched, we have to concatenate all the batches
y_true = np.concatenate([y for x, y in test_ds], axis=0)

In [ ]:
from visualization import summary_graphics


def get_class_training_weights(train_ds, normalize=True):
    labels, counts = np.unique(
        np.concatenate([y for x, y in train_ds], axis=0), return_counts=True
    )
    total = sum(counts)

    weights = [total / (2 * count) for count in counts]
    max_weight = np.max(weights)

    if normalize:
        return {label: weights[i] / max_weight for i, label in enumerate(labels)}

    return {label: weights[i] for i, label in enumerate(labels)}

In [ ]:
class_weight = get_class_training_weights(train_ds=train_ds)

print(f"Weight for normal class: {class_weight[0]:1.3f}")
print(f"Weight for pneumonia class: {class_weight[1]:1.3f}")

## Training

In [ ]:
epochs = 10
model_1 = tf.keras.models.Sequential(
    [
        tf.keras.layers.InputLayer((height, width, channels), name="input"),
        # tf.keras.layers.RandomFlip("horizontal", name="0.1rflip"),
        tf.keras.layers.RandomRotation(0.2, name="0.2rrot"),
        tf.keras.layers.RandomTranslation(0.2, 0.2, name="0.3rtran"),
        tf.keras.layers.RandomZoom(0.2, name="0.4rzoom"),
        tf.keras.layers.Rescaling(1.0 / 255, name="rescale"),
        # initial layers of Xception are similar to VGG16
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.1conv"
        ),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="1.3pool"),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.1conv"
        ),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="2.3pool"),
        # the rest of the convolutional layers in Xception are blocks of 2-3 depthwise separable layers, followed by a pooling layer
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.3conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="3.4pool"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.3conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="4.6pool"),
        # all the CNN models end with dense layers, probably to help the model "interpret" what the convolutional layers see
        tf.keras.layers.Flatten(name="5.1flatten"),
        tf.keras.layers.Dense(1024, activation="relu", name="5.2dense"),
        # the above dense layer has 42 million of the 43 million trainable parameters!!!
        # add in a dropout layer with a high rate to prevent overfitting
        tf.keras.layers.Dropout(0.7, name="5.3dropout"),
        tf.keras.layers.Dense(512, activation="relu", name="5.4dense"),
        tf.keras.layers.Dropout(0.5, name="5.5dropout"),
        # 1 output neuron with sigmoid is equivalent to 2 output neurons with softmax, but trains faster:
        # https://stats.stackexchange.com/questions/207049/neural-network-for-binary-classification-use-1-or-2-output-neurons
        # tf.keras.layers.Dense(2, activation="softmax", name="output")
        tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
    ],
    name="Model_1",
)

metrics = [
    tf.keras.metrics.TruePositives(name="tp"),
    tf.keras.metrics.TrueNegatives(name="tn"),
    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
]

model_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=metrics,
)


def make_callbacks(filepath):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=filepath, save_best_only=True, monitor="val_loss", mode="min"
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min", patience=5, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min", factor=0.5, patience=3
        ),
    ]


model_1.summary()

In [ ]:
history = model_1.fit(
    train_ds,
    validation_data=validation_ds,
    batch_size=batch_size,
    epochs=epochs,
    class_weight=class_weight,
    verbose=1,
    callbacks=make_callbacks("best_model_notransfer_1.keras"),
)

In [ ]:
summary_graphics(history, model_1, test_ds)

In [ ]:
# add in several additional layers in the separable convolution blocks
model_2 = tf.keras.models.Sequential(
    [
        tf.keras.layers.InputLayer((height, width, channels), name="input"),
        # tf.keras.layers.RandomFlip("horizontal", name="0.1rflip"),
        tf.keras.layers.RandomRotation(0.2, name="0.2rrot"),
        tf.keras.layers.RandomTranslation(0.2, 0.2, name="0.3rtran"),
        tf.keras.layers.RandomZoom(0.2, name="0.4rzoom"),
        tf.keras.layers.Rescaling(1.0 / 255, name="rescale"),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.1conv"
        ),
        tf.keras.layers.Conv2D(
            64, (3, 3), activation="relu", padding="same", name="1.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="1.3pool"),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.1conv"
        ),
        tf.keras.layers.Conv2D(
            128, (3, 3), activation="relu", padding="same", name="2.2conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="2.3pool"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.3conv"
        ),
        tf.keras.layers.BatchNormalization(name="3.4batchnorm"),
        tf.keras.layers.SeparableConv2D(
            256, (3, 3), activation="relu", padding="same", name="3.5conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="3.6pool"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.1conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.2batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.3conv"
        ),
        tf.keras.layers.BatchNormalization(name="4.4batchnorm"),
        tf.keras.layers.SeparableConv2D(
            512, (3, 3), activation="relu", padding="same", name="4.5conv"
        ),
        tf.keras.layers.MaxPooling2D((2, 2), name="4.6pool"),
        tf.keras.layers.Flatten(name="5.1flatten"),
        tf.keras.layers.Dense(1024, activation="relu", name="5.2dense"),
        tf.keras.layers.Dropout(0.7, name="5.3dropout"),
        tf.keras.layers.Dense(512, activation="relu", name="5.4dense"),
        tf.keras.layers.Dropout(0.5, name="5.5dropout"),
        tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
    ],
    name="Model_2",
)
# this is now identical to Nain's model (except for the random transformations)

model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=metrics,
)

In [ ]:
history = model_2.fit(
    train_ds,
    validation_data=validation_ds,
    batch_size=batch_size,
    epochs=epochs,
    class_weight=class_weight,
    verbose=1,
    callbacks=make_callbacks("best_model_notransfer_2.keras"),
)

In [ ]:
summary_graphics(history, model_2, test_ds)

In [ ]:
notebook_end_time = datetime.datetime.now()
print(
    f"Notebook last run (end-to-end): {notebook_end_time} (duration: {notebook_end_time - notebook_start_time})"
)